On s'occupe des tables des constructeurs

In [1]:
import numpy as np
import pandas as pd

from src import import_data as i_d
#importation des données

i_d.charger_donnees_depuis_bureau()


#constructor_standing = pd.read_csv("donnees_formule_un/constructor_standings.csv")
#st_res = pd.read_csv("donnees_formule_un/constructor_results.csv")

Dossier trouvé : /Users/gabriels./Desktop/donnees_formule_un
Fichiers CSV trouvés : [PosixPath('/Users/gabriels./Desktop/donnees_formule_un/circuits.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/status.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/lap_times.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/sprint_results.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/drivers.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/races.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/constructors.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/constructor_standings.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/qualifying.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/driver_standings.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/constructor_results.csv'), PosixPath('/Users/gabriels./Desktop/donnees_formule_un/pit_stops.csv'), PosixPath('/Users/gabri

structure de la table constructors :
- constructorID = int
- constructorRef = object
- name = object
- nationality = object
- url = object

structure de la table constructor_results:
- constructorResultsID = int
- raceID = int 
- constructorID = int
- points = float
- status = object

structure de la table constructor_standings:
- constructorStandingsID = int
- raceID = int
- constructorID = int
- points = float
- positionText = object
- wins = int

In [2]:
#recherche des Na : on a des \\N = équivalent
constructors
constructors[constructors.apply(lambda row: row.astype(str).str.contains(r"\\N", na=False).any(), axis=1)]
#pas de \\N

constructor_standings
constructor_standings[constructor_standings.apply(lambda row: row.astype(str).str.contains(r"\\N", na=False).any(), axis=1)]
#pas de \\N

constructor_results
constructor_results[constructor_results.apply(lambda row: row.astype(str).str.contains(r"\\N", na=False).any(), axis=1)]
# un \\N à chaque ligne


#on remplace les \N par des NA:
constructor_results.replace("\\N", np.nan, inplace= True)


races.replace("\\N", np.nan, inplace= True)

**Question : quelle écurie a gagné le plus de courses ? 
Faire un classement des écuries selon le nombre de victoires cumulées.**

In [3]:
#on fait un groubpy pour récupérer le constructorId ayant remporté le plus de points pour chaque course
max_course = constructor_results.loc[:,['constructorId', 'raceId', 'points']].groupby('raceId').agg(premier = ('points', "max"))
#la colonne constructorId n'est pas affiché : on va merge avec d'autres tables pour récupérer les id puis les noms des constructeurs
max_course = pd.merge(max_course, races, on = ["raceId"])
max_course =  max_course[["raceId", "premier"]]
max_course
gagnants = pd.merge(
    max_course,
    constructor_results,
    left_on=["raceId", "premier"],
    right_on=["raceId", "points"], #on fait correspondre "premier" et "points"
    how="inner"
)
gagnants = gagnants[["raceId", "constructorId", "points"]]
gagnants = pd.merge(gagnants, constructors, on = ["constructorId"], how = "inner")
gagnants = gagnants[["raceId", "name", "points"]]
gagnants = gagnants.groupby("name", as_index= False).size()
gagnants.sort_values(by = "size", ascending= False)


,name,size
14,Ferrari,239
25,McLaren,192
42,Williams,126
27,Mercedes,117
31,Red Bull,111
36,Team Lotus,46
32,Renault,34
4,Benetton,28
5,Brabham,24
21,Lotus-Climax,22


La table ci-dessus indique que le contsructeur qui a cumulé le plus de courses remportées d'après la table race est ferrari avec 239 victoires 

**Question 2: quel constructeur a remporté le plus de saisons ? Faire un classement.**

In [4]:
#on veut récupérer la course la plus ancienne de la table race
races["date"].min()
#la première course de la base de donnée a été effectuée le 13/05/1950
season = seasons
season #on remarque que les saisons sont découpées par années et qu'il n'y a pas de "débordement" d'une année à l'autre
#l'écurie qui aura remporté le plus de points dans la saison la remporte
#on veut regrouper à la fois par année et par constructeur

total_course = pd.merge(races, constructor_results, how = "inner")
total_course = total_course.loc[:, ["raceId", "year", "constructorId", "points"]]
total_course

,raceId,year,constructorId,points
0,1,2009,23,18.0
1,1,2009,1,0.0
2,1,2009,7,11.0
3,1,2009,4,4.0
4,1,2009,3,3.0
...,...,...,...,...
12500,1132,2024,117,10.0
12501,1132,2024,3,2.0
12502,1132,2024,215,1.0
12503,1132,2024,15,0.0


In [ ]:
total_course
races.groupby("year")["raceId"].count()
#on peut faire le groupby:

tbl = constructor_results.loc[:,['constructorId', 'raceId', 'points']].groupby('raceId').agg(premier = ('points', "max"))
tbl = pd.merge(constructor_results, tbl, left_on = ["raceId", "points"], right_on= ["raceId", "premier"])
tbl


NameError: name 'cst_res' is not defined

In [313]:

tbl2 = pd.merge(races, tbl, on = "raceId")
tbl2.columns
tbl2 = tbl2[["year", "raceId", "constructorId", "name", "points"]]
tbl2


,year,raceId,constructorId,name,points
0,2009,1,23,Australian Grand Prix,18.0
1,2009,2,23,Malaysian Grand Prix,7.0
2,2009,3,9,Chinese Grand Prix,18.0
3,2009,4,23,Bahrain Grand Prix,14.0
4,2009,5,23,Spanish Grand Prix,18.0
...,...,...,...,...,...
1087,2024,1129,1,Canadian Grand Prix,28.0
1088,2024,1129,131,Canadian Grand Prix,28.0
1089,2024,1130,9,Spanish Grand Prix,29.0
1090,2024,1131,131,Austrian Grand Prix,45.0


In [314]:
tbl2 = tbl2.groupby("year", as_index= False)["constructorId"].value_counts()
tbl2

,year,constructorId,count
0,1958,118,5
1,1958,6,3
2,1958,87,2
3,1959,170,5
4,1959,6,2
...,...,...,...
261,2023,131,1
262,2024,9,6
263,2024,1,3
264,2024,6,2


In [315]:
count_max = tbl2.groupby("year", as_index= False)["count"].max()
count_max
merged = pd.merge(count_max, tbl2, on = ["year"])
merged = merged[merged["count_x"] == merged["count_y"]]
merged = merged[["year", "count_x", "constructorId"]]
merged

,year,count_x,constructorId
0,1958,5,118
3,1959,5,170
7,1960,6,170
11,1961,5,6
13,1962,4,66
...,...,...,...
246,2020,11,131
251,2021,11,9
255,2022,15,9
258,2023,18,9


In [316]:
gagnants_saisons = pd.merge(merged, constructors, on = "constructorId")
total_win_saisons = gagnants_saisons.groupby("name", as_index= False)["year"].size()
total_win_saisons.sort_values(by = "size", ascending= False)


,name,size
4,Ferrari,13
8,McLaren,11
10,Red Bull,9
15,Williams,8
9,Mercedes,7
12,Team Lotus,5
5,Lotus-Climax,3
1,Benetton,2
2,Brabham-Repco,2
3,Cooper-Climax,2


On en conclut que Ferrari cumule le plus de saisons remportées. 